# Complex-Valued FEM — the Helmholtz Equation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/helmholtz.ipynb)

TensorMesh assembles and solves in **complex arithmetic** end to end. We solve
the interior Helmholtz problem

$$-\nabla^2 u - k^2 u = 0 \quad\text{in }(0,1)^2, \qquad u = g \text{ on } \partial\Omega$$

against the analytic plane wave $u = e^{ikx}$, which exercises complex
`point_data`, complex Dirichlet condensation, and the complex sparse solve —
the same machinery the PML and waveguide examples build on.

Docs: [Complex-Valued FEM (Helmholtz)](https://docs.tensor-mesh.com/example_gallery/complex.html) · Source: [`examples/wave/helmholtz/helmholtz.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/wave/helmholtz/helmholtz.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

## Weak form

The assembler is dtype-agnostic — nothing in `forward` mentions complex
numbers. Complexity enters through the coefficient `k_sq` and through
`asm.type(torch.complex128)`.

In [ ]:
import warnings

import torch

# PyTorch notes that complex nn.Module buffers are experimental; the complex
# assembly path is exercised by the test suite, so quiet it for readability.
warnings.filterwarnings("ignore", message=".*Complex modules are a new feature.*")

from tensormesh import Condenser, ElementAssembler, Mesh


class HelmholtzAssembler(ElementAssembler):
    r"""Complex Helmholtz stiffness

    .. math::

        a(u, v) = \int_\Omega \nabla u \cdot \nabla v - k^2 u v \, d\Omega.

    ``k_sq`` arrives as per-node ``point_data`` and may be complex — the same
    mechanism PML coefficients use to model damping.
    """

    def forward(self, gradu, gradv, u, v, k_sq):
        return gradu @ gradv - k_sq * u * v


def u_exact(points, k, dtype):
    """Plane wave ``exp(i k x)``."""
    x = points[..., 0].to(dtype=torch.float64)
    return torch.exp(1j * k * x).to(dtype)

## Solve

The body force is zero: $-\nabla^2 e^{ikx} = k^2 e^{ikx}$ cancels the mass
term exactly, so the plane wave is an exact solution and the boundary data
alone drives the problem.

In [ ]:
K_WAVE = 2 * torch.pi
DTYPE = torch.complex128


def solve_helmholtz(chara_length, k=K_WAVE, dtype=DTYPE):
    """Assemble, condense, and solve; return the solution, mesh, and L2 error."""
    with quiet():
        mesh = Mesh.gen_rectangle(chara_length=chara_length, element_type="tri")
    points = mesh.points.double()

    # Complex per-node coefficient (constant here).
    k_sq = torch.full((mesh.n_points,), k * k + 0j, dtype=dtype)

    asm = HelmholtzAssembler.from_mesh(mesh, quadrature_order=3)
    asm.type(dtype)
    H = asm(points=points, point_data={"k_sq": k_sq})

    # Dirichlet data: the exact plane wave on every boundary node.
    g = u_exact(mesh.points, k=k, dtype=dtype)
    rhs = torch.zeros(mesh.n_points, dtype=dtype)
    condenser = Condenser(mesh.boundary_mask, dirichlet_value=g[mesh.boundary_mask])
    H_inner, rhs_inner = condenser(H, rhs)
    u = condenser.recover(H_inner.solve(rhs_inner))

    diff = u - g
    err_l2 = ((diff.conj() * diff).real.sum().item() / mesh.n_points) ** 0.5
    return u, g, mesh, err_l2

## Convergence

P1 elements give $O(h^2)$ in $L^2$ — the error should drop by roughly 4×
each time the mesh size halves.

In [ ]:
print(f"{'h':<10s} {'n_dofs':<10s} {'L2 error':<14s}")
for h in [0.2, 0.1, 0.05, 0.025]:
    _, _, mesh, err = solve_helmholtz(h)
    print(f"{h:<10.3f} {mesh.n_points:<10d} {err:<14.3e}")

## The solution field

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np

u, g, mesh, err = solve_helmholtz(0.05)
u_np, g_np = u.numpy(), g.numpy()
pts = mesh.points.numpy()
triang = mtri.Triangulation(pts[:, 0], pts[:, 1], mesh.cells["triangle"].numpy())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
panels = [
    ("Re(u)", u_np.real, "RdBu_r"),
    ("Im(u)", u_np.imag, "RdBu_r"),
    (r"$|u - u_{exact}|$", np.abs(u_np - g_np), "viridis"),
]
for ax, (title, data, cmap) in zip(axes, panels):
    tcf = ax.tricontourf(triang, data, levels=20, cmap=cmap)
    ax.set_title(title)
    ax.set_aspect("equal")
    fig.colorbar(tcf, ax=ax, shrink=0.85)
fig.suptitle(f"Helmholtz, k = 2π,  h = 0.05,  L2 error = {err:.2e}", y=1.0)
fig.tight_layout()
plt.show()

## Where to next

- [Open-domain wave problems](https://docs.tensor-mesh.com/example_gallery/open_domain_wave.html) — PML absorbing layers and plane-wave ports built on this same complex path.
- [Phononic crystals](https://docs.tensor-mesh.com/example_gallery/phononic_crystal.html) — Bloch-Floquet band structures.